#### Setup: catalog, schemas, and tables
Run once before any other notebook to create the Unity Catalog for the Palm Oil Spreads project pipeline.

**Creates**
- Catalog: `palm`
- Schemas (medallion layers):
  - `palm.bronze`: raw USDA files, loaded as-is
  - `palm.silver`: cleaned and standardized data
  - `palm.gold`  : dimensional model for reporting
- Volume: `palm.bronze.raw`, raw downloaded USDA files
- Tables:
  - `tbd`: saved download and metadata of the parkway road status and closures page (nps.gov)
**Steps**
1. Create the catalog
2. Create the three medallion schemas
3. Create the bronze tables
4. Create the silver tables
5. Create the gold tables
5. Check: list catalog, schemas, and tables

**Safe to re-run:** every statement uses `IF NOT EXISTS`, so running it again never changes or deletes project tables. This also means any changes to an existing table's columns or comments will not be applied on a rerun; use `ALTER TABLE` / `COMMENT ON`, or drop and recreate the table if changes are needed.

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: CATALOG
-- Creates `palm`, the top-level container for everything in the Palm Oil Spreads project.
--
-- The bronze, silver and gold schemas are created
-- inside the palm catalog in the next cells.
--
-- IF NOT EXISTS makes this safe to rerun: if the catalog is already there,
-- nothing happens, and nothing inside it is changed or deleted.
--
-- Note: creating a catalog needs the CREATE CATALOG permission on the
-- metastore. If this fails with a permissions error, a workspace admin
-- needs to grant it.
-- ---------------------------------------------------------------------------
CREATE CATALOG IF NOT EXISTS palmacker
COMMENT 'Palm oil in packaged spreads: USDA Branded Foods, 2018-2026'

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: SCHEMAS (one per medallion layer)
-- Creates the three schemas inside the brp catalog. Each schema holds one
-- layer of the medallion architecture, and data flows bronze -> silver -> gold:
-- ---------------------------------------------------------------------------
CREATE SCHEMA IF NOT EXISTS palm.bronze
COMMENT 'Raw USDA files loaded as-is, plus load metadata';

CREATE SCHEMA IF NOT EXISTS palm.silver
COMMENT 'Cleaned, typed, and standardized data';

CREATE SCHEMA IF NOT EXISTS palm.gold
COMMENT 'Dimensional model and reporting tables';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: BRONZE VOLUME
-- File storage for raw downloads (USDA Branded Food)
-- Files are filtered before load to reduce size.
-- IF NOT EXISTS: safe to rerun; existing files are left untouched.
-- ---------------------------------------------------------------------------
CREATE VOLUME IF NOT EXISTS palm.bronze.raw_files
COMMENT 'Landing zone for USDA Branded Foods files, filtered locally before upload';

In [0]:
# ---------------------------------------------------------------------------
# PROJECT SETUP: RELEASE FOLDERS
# One landing folder per USDA Branded Foods release (using about one per year).
# Folder names use the pattern release=YYYY-MM so the bronze load can read
# the release label straight from each file's path.
# mkdirs is safe to rerun: existing folders and files are left untouched.
# ---------------------------------------------------------------------------

# Releases in scope: one per year, 2018-2026.
# 2018-07 is the pre-FoodData Central release and uses a different file layout.

RELEASES = [
    "2018-07", "2019-12", "2020-10", "2021-10", "2022-10",
    "2023-10", "2024-10", "2025-12", "2026-10",
]

base = "/Volumes/palm/bronze/raw_files/usda_branded"
for release in RELEASES:
    dbutils.fs.mkdirs(f"{base}/release={release}")

display(dbutils.fs.ls(base))

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- SETUP CHECK: WHAT'S IN THE palm CATALOG
-- Lists every schema and the tables inside it, to confirm the medallion
-- setup (bronze, silver, gold) was created correctly.
--
-- Expected: bronze, silver and gold all appear with tables that have been created
-- ---------------------------------------------------------------------------
SELECT
    schemata.catalog_name AS table_catalog,   -- always palm here
    schemata.schema_name  AS table_schema,    -- bronze, silver, gold
    tables.table_name,                        -- a table in that schema (NULL if it has none)
    tables.table_type                         -- MANAGED (Databricks stores the data), EXTERNAL, VIEW...

-- information_schema is a built-in set of read-only views describing what's
-- in the catalog: `schemata` lists schemas, `tables` lists tables.
FROM palm.information_schema.schemata

-- LEFT JOIN keeps every schema even if it has no tables.
-- (A regular JOIN would hide empty schemas, like a new gold layer.)
LEFT JOIN palm.information_schema.tables
    ON  schemata.catalog_name = tables.table_catalog
    AND schemata.schema_name  = tables.table_schema

-- Hide the two schemas Databricks creates automatically in every catalog:
--   information_schema = the built-in metadata views used above
--   default            = an empty starter schema this project doesn't use
WHERE schemata.schema_name NOT IN ('information_schema', 'default')

-- Group the list by schema, then alphabetically by table.
ORDER BY schemata.schema_name, tables.table_name